# CNN KLIEP density-ratio estimation

This notebook preserves the original patch-size and convolution experiment
variants. Each large experiment cell is self-contained; run only the section
you intend to reproduce. Generated outputs have been removed for a clean Git
history.

Run the configuration cell first. Override its paths with the
`DRE_PROJECT_ROOT`, `DRE_DATA_ROOT`, `DRE_OUTPUT_ROOT`, `DRE_RASTER_DIR`, or
`DRE_LANDMASK_PATH` environment variables when needed.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("DRE_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("DRE_DATA_ROOT", PROJECT_ROOT / "data" / "processed")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DRE_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "dre_approaches")).expanduser().resolve()
RASTER_DIR = Path(os.environ.get("DRE_RASTER_DIR", DATA_ROOT / "covariate_rasters")).expanduser().resolve()
LANDMASK_PATH = Path(os.environ.get("DRE_LANDMASK_PATH", DATA_ROOT / "landmask.tif")).expanduser().resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


# Partial convolution


#### Size 3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_3_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_3_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch3.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


#### Size 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_5")
TEST_DIR = str(DATA_ROOT / "test_patches_5")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_5_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 5  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_5_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch5.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


### Size 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_13_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_13_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch13.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


#### Size 23


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_23_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_23_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch23.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


### Size 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_33_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_33_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch33.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


### Size 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_65_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True

# KLIEP: class meaning
# target p(x) is y=1
# source q(x) is y=0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: bounded score from KLIEP log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logratio_to_bounded_score(logratio_np: np.ndarray) -> np.ndarray:
    return sigmoid_np(logratio_np)


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = logratio_to_bounded_score(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# KLIEP objective + normalization
# =========================================================
def kliep_loss_from_scores(f_pos: torch.Tensor, f_neg: torch.Tensor) -> torch.Tensor:
    """
    KLIEP maximizes: mean(f_pos) - logmeanexp(f_neg)
    so we minimize the negative.
    """
    lme = torch.logsumexp(f_neg, dim=0) - torch.log(
        torch.tensor(f_neg.numel(), device=f_neg.device, dtype=f_neg.dtype)
    )
    return -(f_pos.mean() - lme)


@torch.no_grad()
def compute_logZ_over_indices(model, X_values, X_masks, indices: np.ndarray) -> float:
    """
    logZ = log( mean_{x~q} exp(f(x)) ), estimated over a reference q-set (source/background).
    """
    model.eval()
    idx = np.asarray(indices, dtype=np.int64)
    vals = []
    for start in range(0, len(idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        f = model(xv, xm)  # (B,)
        f = torch.clamp(f, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        vals.append(f.detach())
    f_all = torch.cat(vals, dim=0)
    logZ = (
        torch.logsumexp(f_all, dim=0)
        - torch.log(torch.tensor(f_all.numel(), device=f_all.device, dtype=f_all.dtype))
    ).item()
    return float(logZ)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    In KLIEP usage, the output is f_theta(x) (unnormalized log-weight).
    Normalized log density ratio is log w(x) = f(x) - logZ.
    """

    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.dre_head(z)
        return f


# =========================================================
# Training per fold (KLIEP)
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos = train_idx[y_cv[train_idx] == 1]
    train_neg = train_idx[y_cv[train_idx] == 0]
    val_pos = val_idx[y_cv[val_idx] == 1]
    val_neg = val_idx[y_cv[val_idx] == 0]

    if len(train_pos) == 0 or len(train_neg) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both y=1 (target) and y=0 (source) in TRAIN for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos)}, neg={len(train_neg)}), "
        f"val N={len(val_idx)} (pos={len(val_pos)}, neg={len(val_neg)})"
    )

    # For metrics we evaluate on full split
    ds_tr_all = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=False)
    ds_val_all = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_all = DataLoader(ds_tr_all, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, drop_last=False)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    # For KLIEP optimization we need paired batches (with augmentation)
    ds_tr_pos = PatchDREDataset(X_values, X_masks, y_cv, train_pos, train=True)
    ds_tr_neg = PatchDREDataset(X_values, X_masks, y_cv, train_neg, train=True)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_neg = DataLoader(ds_tr_neg, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)

    steps_per_epoch = min(len(dl_tr_pos), len(dl_tr_neg))
    if steps_per_epoch <= 0:
        raise RuntimeError(f"[fold {fold_id}] Not enough batches for KLIEP training.")

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def cycle_loader(loader):
        while True:
            for batch in loader:
                yield batch

    pos_it = cycle_loader(dl_tr_pos)
    neg_it = cycle_loader(dl_tr_neg)

    @torch.no_grad()
    def eval_with_fixed_logZ(loader_all, logZ_ref: float):
        model.eval()
        all_logratio, all_y = [], []

        for xv, xm, yb in loader_all:
            xv, xm = xv.to(device), xm.to(device)
            f_x = model(xv, xm)
            f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            logratio = f_x - float(logZ_ref)
            all_logratio.append(logratio.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        all_logratio = np.concatenate(all_logratio)
        all_y = np.concatenate(all_y)

        # Reported split-loss: -( mean_{pos} f - logZ ) = -( mean_{pos} (logratio+logZ) - logZ ) = -mean_{pos}(logratio)
        pos_mask = (all_y == 1)
        if pos_mask.sum() > 0:
            loss_val = float(-np.mean(all_logratio[pos_mask]))
        else:
            # no positives -> undefined objective; treat as very bad for selection
            loss_val = 1e9

        mets = compute_epoch_metrics_from_logratio(all_logratio, all_y, loss_val)
        return mets, all_logratio, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        # -------- train --------
        model.train()
        tr_loss_sum = 0.0

        for _ in range(steps_per_epoch):
            xv_p, xm_p, _ = next(pos_it)
            xv_n, xm_n, _ = next(neg_it)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_n, xm_n = xv_n.to(device), xm_n.to(device)

            opt.zero_grad(set_to_none=True)

            f_pos = model(xv_p, xm_p)
            f_neg = model(xv_n, xm_n)
            f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            f_neg = torch.clamp(f_neg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)

            loss = kliep_loss_from_scores(f_pos, f_neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_sum += loss.item()

        tr_loss = tr_loss_sum / steps_per_epoch

        # -------- normalization from TRAIN source (neg) only --------
        logZ_ref = compute_logZ_over_indices(model, X_values, X_masks, train_neg)

        tr_mets, _, _ = eval_with_fixed_logZ(dl_tr_all, logZ_ref)
        va_mets, val_logratio, val_y = eval_with_fixed_logZ(dl_val_all, logZ_ref)
        tr_mets["loss"] = float(tr_loss)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr_mets['loss']:.4f}, AUC {tr_mets['ROC_AUC']:.4f}, Boyce {tr_mets['Boyce']:.4f} | "
            f"val loss {va_mets['loss']:.4f}, AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ_ref": float(logZ_ref),
            }
            candidates.append(
                {
                    "loss": float(va_mets["loss"]),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": val_logratio,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            # early stop on val loss improvement
            if va_mets["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va_mets["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    finite_losses = [c["loss"] for c in candidates if np.isfinite(c["loss"])]
    if len(finite_losses) == 0:
        raise RuntimeError(f"[fold {fold_id}] All candidate losses are non-finite; cannot select model.")
    loss_min = min(finite_losses)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if np.isfinite(c["loss"]) and c["loss"] <= thresh]
    if len(window) == 0:
        window = [min(candidates, key=lambda c: c["loss"])]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logratio = best["val_logratio"]
    val_y = best["val_y"]
    val_score01 = logratio_to_bounded_score(val_logratio)

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (KLIEP normalized log-ratio)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Missing logZ_ref in checkpoint for fold {fid}.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        per_model_logZ.append(float(state["logZ_ref"]))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_x - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (KLIEP + MASK-AWARE CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_65_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_pconv_patch65.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)

# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (mask-aware / partial conv)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims=(32,),
        dropout=0.35,
        clip_logits: Optional[float] = 10.0
    ):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (same as training):
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    mask propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            bias=bias,
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel, stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    """
    Exactly like training code (name kept, but works for any odd patch size).
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    """
    Output is f_theta(x). KLIEP normalized log density ratio is:
      log w(x) = f(x) - logZ_ref
    """
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)

# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)

# -------------------------------------------------------------
# Load ensemble: KLIEP checkpoints contain logZ_ref
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt
      - fold_ids.npy
      - fold_weights_loss.npy

    Each fold checkpoint must include:
      - model_state
      - in_value_channels
      - patch_size
      - emb_dim
      - hidden_dims
      - logZ_ref

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logZ_t: (K,)  (per-fold logZ_ref)
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).all()) or float(np.sum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        weights = np.maximum(weights, 1e-12)
        weights = weights / weights.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read example ckpt for architecture params
    example = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example["in_value_channels"])
    patch_size = int(example.get("patch_size", 0))
    emb_dim = int(example.get("emb_dim", 32))
    hidden_dims = tuple(example.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", patch_size)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ_ref" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ_ref (KLIEP normalization).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # eval() disables dropout anyway
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ_ref"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(logZ_list, dtype=np.float32), dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size

# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)

# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks

# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (KLIEP: per-fold logZ_ref)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )

    print(f"[Model] patch_size from checkpoint = {patch_size}")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model KLIEP normalized log-ratio: log_r_k = f(x) - logZ_ref_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()

if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Standard convolution


# Patch size: 3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_3_ens")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Ensemble choice: weighted average of log-ratios
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# KLIEP score mapping: log w(x) -> bounded score
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = sigmoid_np(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val) if np.isfinite(loss_val) else float("nan")
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchClassDataset(Dataset):
    """
    Dataset over a fixed index set. Used to create separate loaders for positives (y=1) and background (y=0).
    """
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # output is f(x) (unconstrained real)
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    Ratio is defined by w(x) = exp(f(x)) / E_bg[exp(f(bg))].
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# =========================================================
# KLIEP training per fold
# =========================================================
def train_and_save_fold_model_kliep_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos_idx = train_idx[y_cv[train_idx] == 1]
    train_bg_idx  = train_idx[y_cv[train_idx] == 0]

    val_pos_idx = val_idx[y_cv[val_idx] == 1]
    val_bg_idx  = val_idx[y_cv[val_idx] == 0]

    if len(train_pos_idx) < 2 or len(train_bg_idx) < 2:
        raise RuntimeError(f"[fold {fold_id}] Not enough pos/bg samples for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos_idx)}, bg={len(train_bg_idx)}), "
        f"val N={len(val_idx)} (pos={len(val_pos_idx)}, bg={len(val_bg_idx)})"
    )

    ds_tr_pos = PatchClassDataset(X_values, X_masks, y_cv, train_pos_idx, train=True)
    ds_tr_bg  = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx,  train=True)
    ds_val_all = PatchClassDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_bg  = DataLoader(ds_tr_bg,  batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKLIEP(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def kliep_loss(f_pos: torch.Tensor, f_bg: torch.Tensor) -> torch.Tensor:
        # negative KLIEP objective: -( mean_pos f - log mean_bg exp(f_bg) )
        f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        f_bg  = torch.clamp(f_bg,  -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        logZ = torch.log(torch.mean(torch.exp(f_bg)) + EPS)
        return -(torch.mean(f_pos) - logZ)

    def compute_logZ_over_train_bg() -> float:
        # logZ = log(E_bg exp(f(bg))) estimated over ALL training bg samples
        model.eval()
        ds_bg_eval = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx, train=False)
        dl_bg_eval = DataLoader(ds_bg_eval, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, drop_last=False)
        vals = []
        with torch.no_grad():
            for xv, xm, _ in dl_bg_eval:
                xv, xm = xv.to(device), xm.to(device)
                f_bg = model(xv, xm)
                f_bg = torch.clamp(f_bg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                vals.append(torch.exp(f_bg).detach().cpu())
        vals = torch.cat(vals, dim=0)
        logZ = torch.log(torch.mean(vals) + EPS).item()
        return float(logZ)

    def eval_val_all(logZ: float):
        model.eval()
        all_f, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val_all:
                xv, xm = xv.to(device), xm.to(device)
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                all_f.append(f_x.cpu().numpy())
                all_y.append(yb.numpy())
        all_f = np.concatenate(all_f)
        all_y = np.concatenate(all_y)

        logratio = np.clip(all_f - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        mets = compute_epoch_metrics_from_logratio(logratio, all_y, loss_val=np.nan)
        return mets, logratio, all_y

    # selection buffers
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        # epoch length: minimum of the two loader lengths
        n_batches = min(len(dl_tr_pos), len(dl_tr_bg))
        pos_iter = iter(dl_tr_pos)
        bg_iter = iter(dl_tr_bg)

        for _ in range(n_batches):
            xv_p, xm_p, _ = next(pos_iter)
            xv_b, xm_b, _ = next(bg_iter)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_b, xm_b = xv_b.to(device), xm_b.to(device)

            opt.zero_grad(set_to_none=True)
            f_pos = model(xv_p, xm_p)
            f_bg  = model(xv_b, xm_b)
            loss = kliep_loss(f_pos, f_bg)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        tr_loss = total_loss / max(1, steps)

        # full-train background normalizer for evaluation / checkpointing
        logZ = compute_logZ_over_train_bg()

        va_mets, va_logratio, va_y = eval_val_all(logZ)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train KLIEP loss {tr_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f} | "
            f"logZ {logZ:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ": float(logZ),
            }
            candidates.append(
                {
                    "train_loss": float(tr_loss),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": va_logratio,
                    "val_y": va_y,
                    "epoch": ep,
                }
            )

            if tr_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(tr_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["train_loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["train_loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["train_loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"train_loss={best['train_loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # OOF bounded scores on val subset
    val_logratio = best["val_logratio"]
    val_score01 = sigmoid_np(val_logratio)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["train_loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction for KLIEP checkpoints
# =========================================================
def ensemble_predict_on_indices_kliep_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Missing logZ in checkpoint for fold {fid}")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()

        models.append(model)
        per_model_logZ.append(float(state["logZ"]))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_k = model(xv, xm)
                f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_k - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kliep_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen train losses:", fold_losses)

    # ----------- Loss-based weights (lower loss -> higher weight) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_kliep_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — set these to your paths
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_3_ens")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_kliep_suitability_3_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Numeric stability (must match training)
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Output control
# - If True: write log-ratio map (f(x) - logZ) clamped
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode (match training script)
ENSEMBLE_IN_LOGSPACE = True

# Training-time architecture hyperparams (kept as defaults; ckpt may override)
EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — must match training
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # for 3x3 this is odd, but you trained with PATCH_SIZE=3
                                 # If you actually trained on 13x13 patches, set PATCH_SIZE accordingly.
                                 # Keeping your training code as-is.

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    log-ratio used at inference: log w(x) = f(x) - logZ
    where logZ = log(E_bg exp(f(bg))) stored per fold in checkpoint.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# -------------------------------------------------------------
# Stable sigmoid for bounded score from log-ratio
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load KLIEP ensemble: fold checkpoints contain fold-specific logZ
# -------------------------------------------------------------
def load_kliep_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNKLIEP], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_kliep_fold_<fid>.pt  (each contains logZ)
      - fold_ids.npy
      - fold_weights_loss.npy    (or you can swap to fold_weights_loss.npy name used in training)
    Returns:
      models: List[CNNKLIEP]
      weights_t: (K,)
      logZ_t: (K,) where logZ_t[k] is from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_kliep_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", EMB_DIM))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", HIDDEN_DIMS))

    models: List[CNNKLIEP] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_kliep_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ.")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,  # eval() disables dropout anyway
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(logZ_list, dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters (KLIEP ensemble)
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load KLIEP ensemble (logZ per fold)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_kliep_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = f_k(x) - logZ_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "KLIEP log-ratio log_w = f - logZ (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_w) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# patch size:5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_23_ens")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Ensemble choice: weighted average of log-ratios
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# KLIEP score mapping: log w(x) -> bounded score
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = sigmoid_np(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val) if np.isfinite(loss_val) else float("nan")
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchClassDataset(Dataset):
    """
    Dataset over a fixed index set. Used to create separate loaders for positives (y=1) and background (y=0).
    """
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # output is f(x) (unconstrained real)
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    Ratio is defined by w(x) = exp(f(x)) / E_bg[exp(f(bg))].
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# =========================================================
# KLIEP training per fold
# =========================================================
def train_and_save_fold_model_kliep_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos_idx = train_idx[y_cv[train_idx] == 1]
    train_bg_idx  = train_idx[y_cv[train_idx] == 0]

    val_pos_idx = val_idx[y_cv[val_idx] == 1]
    val_bg_idx  = val_idx[y_cv[val_idx] == 0]

    if len(train_pos_idx) < 2 or len(train_bg_idx) < 2:
        raise RuntimeError(f"[fold {fold_id}] Not enough pos/bg samples for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos_idx)}, bg={len(train_bg_idx)}), "
        f"val N={len(val_idx)} (pos={len(val_pos_idx)}, bg={len(val_bg_idx)})"
    )

    ds_tr_pos = PatchClassDataset(X_values, X_masks, y_cv, train_pos_idx, train=True)
    ds_tr_bg  = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx,  train=True)
    ds_val_all = PatchClassDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_bg  = DataLoader(ds_tr_bg,  batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKLIEP(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def kliep_loss(f_pos: torch.Tensor, f_bg: torch.Tensor) -> torch.Tensor:
        # negative KLIEP objective: -( mean_pos f - log mean_bg exp(f_bg) )
        f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        f_bg  = torch.clamp(f_bg,  -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        logZ = torch.log(torch.mean(torch.exp(f_bg)) + EPS)
        return -(torch.mean(f_pos) - logZ)

    def compute_logZ_over_train_bg() -> float:
        # logZ = log(E_bg exp(f(bg))) estimated over ALL training bg samples
        model.eval()
        ds_bg_eval = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx, train=False)
        dl_bg_eval = DataLoader(ds_bg_eval, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, drop_last=False)
        vals = []
        with torch.no_grad():
            for xv, xm, _ in dl_bg_eval:
                xv, xm = xv.to(device), xm.to(device)
                f_bg = model(xv, xm)
                f_bg = torch.clamp(f_bg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                vals.append(torch.exp(f_bg).detach().cpu())
        vals = torch.cat(vals, dim=0)
        logZ = torch.log(torch.mean(vals) + EPS).item()
        return float(logZ)

    def eval_val_all(logZ: float):
        model.eval()
        all_f, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val_all:
                xv, xm = xv.to(device), xm.to(device)
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                all_f.append(f_x.cpu().numpy())
                all_y.append(yb.numpy())
        all_f = np.concatenate(all_f)
        all_y = np.concatenate(all_y)

        logratio = np.clip(all_f - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        mets = compute_epoch_metrics_from_logratio(logratio, all_y, loss_val=np.nan)
        return mets, logratio, all_y

    # selection buffers
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        # epoch length: minimum of the two loader lengths
        n_batches = min(len(dl_tr_pos), len(dl_tr_bg))
        pos_iter = iter(dl_tr_pos)
        bg_iter = iter(dl_tr_bg)

        for _ in range(n_batches):
            xv_p, xm_p, _ = next(pos_iter)
            xv_b, xm_b, _ = next(bg_iter)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_b, xm_b = xv_b.to(device), xm_b.to(device)

            opt.zero_grad(set_to_none=True)
            f_pos = model(xv_p, xm_p)
            f_bg  = model(xv_b, xm_b)
            loss = kliep_loss(f_pos, f_bg)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        tr_loss = total_loss / max(1, steps)

        # full-train background normalizer for evaluation / checkpointing
        logZ = compute_logZ_over_train_bg()

        va_mets, va_logratio, va_y = eval_val_all(logZ)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train KLIEP loss {tr_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f} | "
            f"logZ {logZ:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ": float(logZ),
            }
            candidates.append(
                {
                    "train_loss": float(tr_loss),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": va_logratio,
                    "val_y": va_y,
                    "epoch": ep,
                }
            )

            if tr_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(tr_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["train_loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["train_loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["train_loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"train_loss={best['train_loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # OOF bounded scores on val subset
    val_logratio = best["val_logratio"]
    val_score01 = sigmoid_np(val_logratio)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["train_loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction for KLIEP checkpoints
# =========================================================
def ensemble_predict_on_indices_kliep_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Missing logZ in checkpoint for fold {fid}")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()

        models.append(model)
        per_model_logZ.append(float(state["logZ"]))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_k = model(xv, xm)
                f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_k - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kliep_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen train losses:", fold_losses)

    # ----------- Loss-based weights (lower loss -> higher weight) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_kliep_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_5_ens")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_suitability_kliep_5_logspace_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 5   # <-- MUST match what your raster patch extraction produces
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING (KLIEP)
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)  # f(x)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNKLIEP(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32, hidden_dims=(32,), dropout: float = 0.35):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.f_head(z)  # f(x)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load KLIEP ensemble: fold checkpoints contain logZ
# -------------------------------------------------------------
def load_cnn_ensemble_models_kliep(model_dir: str) -> Tuple[List[CNNKLIEP], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_kliep_fold_<fid>.pt  (each contains logZ)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNKLIEP]
      weights_t: (K,)
      logZ_t: (K,) per fold
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).any()) or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_kliep_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNKLIEP] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_kliep_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ.")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(logZ_list, dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters (KLIEP)
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load KLIEP ensemble
    models, weights_t, logZ_t, in_value_channels, patch_size = load_cnn_ensemble_models_kliep(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log w_k(x) = f_k(x) - logZ_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log w(x) (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log w(x)) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_13_ens")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Ensemble choice: weighted average of log-ratios
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# KLIEP score mapping: log w(x) -> bounded score
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = sigmoid_np(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val) if np.isfinite(loss_val) else float("nan")
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchClassDataset(Dataset):
    """
    Dataset over a fixed index set. Used to create separate loaders for positives (y=1) and background (y=0).
    """
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # output is f(x) (unconstrained real)
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    Ratio is defined by w(x) = exp(f(x)) / E_bg[exp(f(bg))].
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# =========================================================
# KLIEP training per fold
# =========================================================
def train_and_save_fold_model_kliep_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos_idx = train_idx[y_cv[train_idx] == 1]
    train_bg_idx  = train_idx[y_cv[train_idx] == 0]

    val_pos_idx = val_idx[y_cv[val_idx] == 1]
    val_bg_idx  = val_idx[y_cv[val_idx] == 0]

    if len(train_pos_idx) < 2 or len(train_bg_idx) < 2:
        raise RuntimeError(f"[fold {fold_id}] Not enough pos/bg samples for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos_idx)}, bg={len(train_bg_idx)}), "
        f"val N={len(val_idx)} (pos={len(val_pos_idx)}, bg={len(val_bg_idx)})"
    )

    ds_tr_pos = PatchClassDataset(X_values, X_masks, y_cv, train_pos_idx, train=True)
    ds_tr_bg  = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx,  train=True)
    ds_val_all = PatchClassDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_bg  = DataLoader(ds_tr_bg,  batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKLIEP(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def kliep_loss(f_pos: torch.Tensor, f_bg: torch.Tensor) -> torch.Tensor:
        # negative KLIEP objective: -( mean_pos f - log mean_bg exp(f_bg) )
        f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        f_bg  = torch.clamp(f_bg,  -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        logZ = torch.log(torch.mean(torch.exp(f_bg)) + EPS)
        return -(torch.mean(f_pos) - logZ)

    def compute_logZ_over_train_bg() -> float:
        # logZ = log(E_bg exp(f(bg))) estimated over ALL training bg samples
        model.eval()
        ds_bg_eval = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx, train=False)
        dl_bg_eval = DataLoader(ds_bg_eval, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, drop_last=False)
        vals = []
        with torch.no_grad():
            for xv, xm, _ in dl_bg_eval:
                xv, xm = xv.to(device), xm.to(device)
                f_bg = model(xv, xm)
                f_bg = torch.clamp(f_bg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                vals.append(torch.exp(f_bg).detach().cpu())
        vals = torch.cat(vals, dim=0)
        logZ = torch.log(torch.mean(vals) + EPS).item()
        return float(logZ)

    def eval_val_all(logZ: float):
        model.eval()
        all_f, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val_all:
                xv, xm = xv.to(device), xm.to(device)
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                all_f.append(f_x.cpu().numpy())
                all_y.append(yb.numpy())
        all_f = np.concatenate(all_f)
        all_y = np.concatenate(all_y)

        logratio = np.clip(all_f - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        mets = compute_epoch_metrics_from_logratio(logratio, all_y, loss_val=np.nan)
        return mets, logratio, all_y

    # selection buffers
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        # epoch length: minimum of the two loader lengths
        n_batches = min(len(dl_tr_pos), len(dl_tr_bg))
        pos_iter = iter(dl_tr_pos)
        bg_iter = iter(dl_tr_bg)

        for _ in range(n_batches):
            xv_p, xm_p, _ = next(pos_iter)
            xv_b, xm_b, _ = next(bg_iter)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_b, xm_b = xv_b.to(device), xm_b.to(device)

            opt.zero_grad(set_to_none=True)
            f_pos = model(xv_p, xm_p)
            f_bg  = model(xv_b, xm_b)
            loss = kliep_loss(f_pos, f_bg)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        tr_loss = total_loss / max(1, steps)

        # full-train background normalizer for evaluation / checkpointing
        logZ = compute_logZ_over_train_bg()

        va_mets, va_logratio, va_y = eval_val_all(logZ)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train KLIEP loss {tr_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f} | "
            f"logZ {logZ:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ": float(logZ),
            }
            candidates.append(
                {
                    "train_loss": float(tr_loss),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": va_logratio,
                    "val_y": va_y,
                    "epoch": ep,
                }
            )

            if tr_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(tr_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["train_loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["train_loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["train_loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"train_loss={best['train_loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # OOF bounded scores on val subset
    val_logratio = best["val_logratio"]
    val_score01 = sigmoid_np(val_logratio)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["train_loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction for KLIEP checkpoints
# =========================================================
def ensemble_predict_on_indices_kliep_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Missing logZ in checkpoint for fold {fid}")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()

        models.append(model)
        per_model_logZ.append(float(state["logZ"]))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_k = model(xv, xm)
                f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_k - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kliep_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen train losses:", fold_losses)

    # ----------- Loss-based weights (lower loss -> higher weight) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_kliep_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


# Patch Size: 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_33_ens")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Ensemble choice: weighted average of log-ratios
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# KLIEP score mapping: log w(x) -> bounded score
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = sigmoid_np(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val) if np.isfinite(loss_val) else float("nan")
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchClassDataset(Dataset):
    """
    Dataset over a fixed index set. Used to create separate loaders for positives (y=1) and background (y=0).
    """
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # output is f(x) (unconstrained real)
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    Ratio is defined by w(x) = exp(f(x)) / E_bg[exp(f(bg))].
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# =========================================================
# KLIEP training per fold
# =========================================================
def train_and_save_fold_model_kliep_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos_idx = train_idx[y_cv[train_idx] == 1]
    train_bg_idx  = train_idx[y_cv[train_idx] == 0]

    val_pos_idx = val_idx[y_cv[val_idx] == 1]
    val_bg_idx  = val_idx[y_cv[val_idx] == 0]

    if len(train_pos_idx) < 2 or len(train_bg_idx) < 2:
        raise RuntimeError(f"[fold {fold_id}] Not enough pos/bg samples for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos_idx)}, bg={len(train_bg_idx)}), "
        f"val N={len(val_idx)} (pos={len(val_pos_idx)}, bg={len(val_bg_idx)})"
    )

    ds_tr_pos = PatchClassDataset(X_values, X_masks, y_cv, train_pos_idx, train=True)
    ds_tr_bg  = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx,  train=True)
    ds_val_all = PatchClassDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_bg  = DataLoader(ds_tr_bg,  batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKLIEP(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def kliep_loss(f_pos: torch.Tensor, f_bg: torch.Tensor) -> torch.Tensor:
        # negative KLIEP objective: -( mean_pos f - log mean_bg exp(f_bg) )
        f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        f_bg  = torch.clamp(f_bg,  -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        logZ = torch.log(torch.mean(torch.exp(f_bg)) + EPS)
        return -(torch.mean(f_pos) - logZ)

    def compute_logZ_over_train_bg() -> float:
        # logZ = log(E_bg exp(f(bg))) estimated over ALL training bg samples
        model.eval()
        ds_bg_eval = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx, train=False)
        dl_bg_eval = DataLoader(ds_bg_eval, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, drop_last=False)
        vals = []
        with torch.no_grad():
            for xv, xm, _ in dl_bg_eval:
                xv, xm = xv.to(device), xm.to(device)
                f_bg = model(xv, xm)
                f_bg = torch.clamp(f_bg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                vals.append(torch.exp(f_bg).detach().cpu())
        vals = torch.cat(vals, dim=0)
        logZ = torch.log(torch.mean(vals) + EPS).item()
        return float(logZ)

    def eval_val_all(logZ: float):
        model.eval()
        all_f, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val_all:
                xv, xm = xv.to(device), xm.to(device)
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                all_f.append(f_x.cpu().numpy())
                all_y.append(yb.numpy())
        all_f = np.concatenate(all_f)
        all_y = np.concatenate(all_y)

        logratio = np.clip(all_f - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        mets = compute_epoch_metrics_from_logratio(logratio, all_y, loss_val=np.nan)
        return mets, logratio, all_y

    # selection buffers
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        # epoch length: minimum of the two loader lengths
        n_batches = min(len(dl_tr_pos), len(dl_tr_bg))
        pos_iter = iter(dl_tr_pos)
        bg_iter = iter(dl_tr_bg)

        for _ in range(n_batches):
            xv_p, xm_p, _ = next(pos_iter)
            xv_b, xm_b, _ = next(bg_iter)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_b, xm_b = xv_b.to(device), xm_b.to(device)

            opt.zero_grad(set_to_none=True)
            f_pos = model(xv_p, xm_p)
            f_bg  = model(xv_b, xm_b)
            loss = kliep_loss(f_pos, f_bg)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        tr_loss = total_loss / max(1, steps)

        # full-train background normalizer for evaluation / checkpointing
        logZ = compute_logZ_over_train_bg()

        va_mets, va_logratio, va_y = eval_val_all(logZ)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train KLIEP loss {tr_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f} | "
            f"logZ {logZ:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ": float(logZ),
            }
            candidates.append(
                {
                    "train_loss": float(tr_loss),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": va_logratio,
                    "val_y": va_y,
                    "epoch": ep,
                }
            )

            if tr_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(tr_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["train_loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["train_loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["train_loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"train_loss={best['train_loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # OOF bounded scores on val subset
    val_logratio = best["val_logratio"]
    val_score01 = sigmoid_np(val_logratio)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["train_loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction for KLIEP checkpoints
# =========================================================
def ensemble_predict_on_indices_kliep_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Missing logZ in checkpoint for fold {fid}")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()

        models.append(model)
        per_model_logZ.append(float(state["logZ"]))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_k = model(xv, xm)
                f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_k - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kliep_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen train losses:", fold_losses)

    # ----------- Loss-based weights (lower loss -> higher weight) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_kliep_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — set these to your paths
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_33_ens")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_kliep_suitability_33_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 33
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Numeric stability (must match training)
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Output control
# - If True: write log-ratio map (f(x) - logZ) clamped
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode (match training script)
ENSEMBLE_IN_LOGSPACE = True

# Training-time architecture hyperparams (kept as defaults; ckpt may override)
EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — must match training
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # for 3x3 this is odd, but you trained with PATCH_SIZE=3
                                 # If you actually trained on 13x13 patches, set PATCH_SIZE accordingly.
                                 # Keeping your training code as-is.

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    log-ratio used at inference: log w(x) = f(x) - logZ
    where logZ = log(E_bg exp(f(bg))) stored per fold in checkpoint.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# -------------------------------------------------------------
# Stable sigmoid for bounded score from log-ratio
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load KLIEP ensemble: fold checkpoints contain fold-specific logZ
# -------------------------------------------------------------
def load_kliep_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNKLIEP], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_kliep_fold_<fid>.pt  (each contains logZ)
      - fold_ids.npy
      - fold_weights_loss.npy    (or you can swap to fold_weights_loss.npy name used in training)
    Returns:
      models: List[CNNKLIEP]
      weights_t: (K,)
      logZ_t: (K,) where logZ_t[k] is from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_kliep_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", EMB_DIM))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", HIDDEN_DIMS))

    models: List[CNNKLIEP] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_kliep_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ.")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,  # eval() disables dropout anyway
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(logZ_list, dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters (KLIEP ensemble)
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load KLIEP ensemble (logZ per fold)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_kliep_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = f_k(x) - logZ_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "KLIEP log-ratio log_w = f - logZ (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_w) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_65_ens")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Ensemble choice: weighted average of log-ratios
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# KLIEP score mapping: log w(x) -> bounded score
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics_from_logratio(logratio_np, y_np, loss_val):
    score01 = sigmoid_np(logratio_np)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val) if np.isfinite(loss_val) else float("nan")
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchClassDataset(Dataset):
    """
    Dataset over a fixed index set. Used to create separate loaders for positives (y=1) and background (y=0).
    """
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # output is f(x) (unconstrained real)
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    Ratio is defined by w(x) = exp(f(x)) / E_bg[exp(f(bg))].
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# =========================================================
# KLIEP training per fold
# =========================================================
def train_and_save_fold_model_kliep_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    train_pos_idx = train_idx[y_cv[train_idx] == 1]
    train_bg_idx  = train_idx[y_cv[train_idx] == 0]

    val_pos_idx = val_idx[y_cv[val_idx] == 1]
    val_bg_idx  = val_idx[y_cv[val_idx] == 0]

    if len(train_pos_idx) < 2 or len(train_bg_idx) < 2:
        raise RuntimeError(f"[fold {fold_id}] Not enough pos/bg samples for KLIEP.")

    print(
        f"\n[fold {fold_id}] train N={len(train_idx)} (pos={len(train_pos_idx)}, bg={len(train_bg_idx)}), "
        f"val N={len(val_idx)} (pos={len(val_pos_idx)}, bg={len(val_bg_idx)})"
    )

    ds_tr_pos = PatchClassDataset(X_values, X_masks, y_cv, train_pos_idx, train=True)
    ds_tr_bg  = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx,  train=True)
    ds_val_all = PatchClassDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr_pos = DataLoader(ds_tr_pos, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_tr_bg  = DataLoader(ds_tr_bg,  batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
    dl_val_all = DataLoader(ds_val_all, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKLIEP(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def kliep_loss(f_pos: torch.Tensor, f_bg: torch.Tensor) -> torch.Tensor:
        # negative KLIEP objective: -( mean_pos f - log mean_bg exp(f_bg) )
        f_pos = torch.clamp(f_pos, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        f_bg  = torch.clamp(f_bg,  -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        logZ = torch.log(torch.mean(torch.exp(f_bg)) + EPS)
        return -(torch.mean(f_pos) - logZ)

    def compute_logZ_over_train_bg() -> float:
        # logZ = log(E_bg exp(f(bg))) estimated over ALL training bg samples
        model.eval()
        ds_bg_eval = PatchClassDataset(X_values, X_masks, y_cv, train_bg_idx, train=False)
        dl_bg_eval = DataLoader(ds_bg_eval, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, drop_last=False)
        vals = []
        with torch.no_grad():
            for xv, xm, _ in dl_bg_eval:
                xv, xm = xv.to(device), xm.to(device)
                f_bg = model(xv, xm)
                f_bg = torch.clamp(f_bg, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                vals.append(torch.exp(f_bg).detach().cpu())
        vals = torch.cat(vals, dim=0)
        logZ = torch.log(torch.mean(vals) + EPS).item()
        return float(logZ)

    def eval_val_all(logZ: float):
        model.eval()
        all_f, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val_all:
                xv, xm = xv.to(device), xm.to(device)
                f_x = model(xv, xm)
                f_x = torch.clamp(f_x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                all_f.append(f_x.cpu().numpy())
                all_y.append(yb.numpy())
        all_f = np.concatenate(all_f)
        all_y = np.concatenate(all_y)

        logratio = np.clip(all_f - logZ, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
        mets = compute_epoch_metrics_from_logratio(logratio, all_y, loss_val=np.nan)
        return mets, logratio, all_y

    # selection buffers
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        steps = 0

        # epoch length: minimum of the two loader lengths
        n_batches = min(len(dl_tr_pos), len(dl_tr_bg))
        pos_iter = iter(dl_tr_pos)
        bg_iter = iter(dl_tr_bg)

        for _ in range(n_batches):
            xv_p, xm_p, _ = next(pos_iter)
            xv_b, xm_b, _ = next(bg_iter)

            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_b, xm_b = xv_b.to(device), xm_b.to(device)

            opt.zero_grad(set_to_none=True)
            f_pos = model(xv_p, xm_p)
            f_bg  = model(xv_b, xm_b)
            loss = kliep_loss(f_pos, f_bg)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        tr_loss = total_loss / max(1, steps)

        # full-train background normalizer for evaluation / checkpointing
        logZ = compute_logZ_over_train_bg()

        va_mets, va_logratio, va_y = eval_val_all(logZ)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train KLIEP loss {tr_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f} | "
            f"logZ {logZ:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "logZ": float(logZ),
            }
            candidates.append(
                {
                    "train_loss": float(tr_loss),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logratio": va_logratio,
                    "val_y": va_y,
                    "epoch": ep,
                }
            )

            if tr_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(tr_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["train_loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["train_loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["train_loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"train_loss={best['train_loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # OOF bounded scores on val subset
    val_logratio = best["val_logratio"]
    val_score01 = sigmoid_np(val_logratio)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["train_loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction for KLIEP checkpoints
# =========================================================
def ensemble_predict_on_indices_kliep_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logZ = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kliep_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Missing logZ in checkpoint for fold {fid}")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()

        models.append(model)
        per_model_logZ.append(float(state["logZ"]))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(np.asarray(per_model_logZ, dtype=np.float32), device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                f_k = model(xv, xm)
                f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                log_ratio_k = f_k - logZ_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kliep_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen train losses:", fold_losses)

    # ----------- Loss-based weights (lower loss -> higher weight) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_kliep_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — set these to your paths
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "kliep" / "cnn_kliep_patch_models_3_ens")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "kliep" / "dengue_cnn_kliep_suitability_3_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Numeric stability (must match training)
MAX_ABS_LOG_RATIO = 50.0
EPS = 1e-12

# Output control
# - If True: write log-ratio map (f(x) - logZ) clamped
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode (match training script)
ENSEMBLE_IN_LOGSPACE = True

# Training-time architecture hyperparams (kept as defaults; ckpt may override)
EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — must match training
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # for 3x3 this is odd, but you trained with PATCH_SIZE=3
                                 # If you actually trained on 13x13 patches, set PATCH_SIZE accordingly.
                                 # Keeping your training code as-is.

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNKLIEP(nn.Module):
    """
    KLIEP ratio model: f(x) produced by network.
    log-ratio used at inference: log w(x) = f(x) - logZ
    where logZ = log(E_bg exp(f(bg))) stored per fold in checkpoint.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.f_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        f = self.f_head(z)
        return f


# -------------------------------------------------------------
# Stable sigmoid for bounded score from log-ratio
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load KLIEP ensemble: fold checkpoints contain fold-specific logZ
# -------------------------------------------------------------
def load_kliep_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNKLIEP], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_kliep_fold_<fid>.pt  (each contains logZ)
      - fold_ids.npy
      - fold_weights_loss.npy    (or you can swap to fold_weights_loss.npy name used in training)
    Returns:
      models: List[CNNKLIEP]
      weights_t: (K,)
      logZ_t: (K,) where logZ_t[k] is from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_kliep_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", EMB_DIM))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", HIDDEN_DIMS))

    models: List[CNNKLIEP] = []
    logZ_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_kliep_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if "logZ" not in state:
            raise RuntimeError(f"Checkpoint for fold {fid} missing logZ.")

        model = CNNKLIEP(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,  # eval() disables dropout anyway
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        logZ_list.append(float(state["logZ"]))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logZ_t = torch.tensor(logZ_list, dtype=torch.float32, device=device)

    return models, weights_t, logZ_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters (KLIEP ensemble)
# -------------------------------------------------------------
def predict_suitability_map_cnn_kliep(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load KLIEP ensemble (logZ per fold)
    models, weights_t, logZ_t, in_value_channels, patch_size = load_kliep_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio map, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = f_k(x) - logZ_k
                        logratios = []
                        for k, model in enumerate(models):
                            f_k = model(xv, xm)  # (B,)
                            f_k = torch.clamp(f_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            log_r_k = f_k - logZ_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "KLIEP log-ratio log_w = f - logZ (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_w) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn_kliep(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )
